In [ ]:
import hashlib
import os
import sys
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
MODEL = os.getenv("MODEL")
BASE_URL = os.getenv("OLLAMA_BASE_URL")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "wraith").is_dir():
	REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))

from wraith.parsers.cowrie_parser import (
	resolve_log_files,
	load_events_df,
	build_session_summaries_df,
	dataframe_to_summaries,
	parse_sessions
)
from wraith.enrichment.session_stats import (
	credential_stats,
	ip_stats,
	command_stats,
	download_stats,
	technique_stats
)
from wraith.llm.session_assessment import analyze_session
from wraith.enrichment.triage import iter_triage
from wraith.audit.audit_log import log_batch, new_run_context

## 0. Verify the parser against a safe fixture

This deliberately small fixture includes a sparse connect/close session, a Cowrie `destfile` download, and one malformed line. It confirms the parser boundary before using live sensor data.

In [2]:
FIXTURE = REPO_ROOT / "data" / "raw" / "fixtures" / "cowrie_sparse.jsonl"

fixture_events_df = load_events_df(FIXTURE)
fixture_sessions_df = build_session_summaries_df(fixture_events_df)

assert len(fixture_events_df) == 7
assert len(fixture_sessions_df) == 2
assert fixture_sessions_df["files_downloaded"].str.len().sum() == 1

print("Parser fixture checks passed")
fixture_sessions_df

Parser fixture checks passed


,session_id,src_ip,start_time,end_time,duration_seconds,login_attempts,login_success,attempted_credentials,successful_credentials,commands,files_downloaded,event_count
0,fixture-noise,192.0.2.10,2026-08-25T10:00:00Z,2026-08-25T10:00:00.2Z,0.2,0,False,[],NaN,[],[],2
1,fixture-download,198.51.100.20,2026-08-25T10:01:00Z,2026-08-25T10:01:04Z,4.0,1,True,[root:example-fixture-only],root:example-fixture-only,[wget https://example.invalid/payload -O /tmp/...,[/tmp/payload],5


## 1. Pick file(s)

Set `TARGET` to exactly one of:

- an exact file path -> that one file, alone
- a directory path -> every `cowrie.json*` file inside it
- a wildcard pattern string, e.g. `"cowrie.json.2026-08-2*"` -> every
  file matching that pattern

No date parsing involved - it's pure filename matching.

In [ ]:
RAW_LOG_DIR = REPO_ROOT / "data" / "raw"

TARGET = RAW_LOG_DIR / "cowrie.json.2026-08-20"
# TARGET = RAW_LOG_DIR
# TARGET = str(RAW_LOG_DIR / "cowrie.json.2026-08-2*")

log_files = resolve_log_files(TARGET)
print(f"{len(log_files)} file(s) selected: {[f.name for f in log_files]}")

1 file(s) selected: ['cowrie.json.2026-08-23']


## 2. Parse

`parse_sessions()` prints a stats overview automatically
(`verbose=True` by default) - credentials, source IPs, commands,
downloads - straight from the event-level data. Returns a
`SessionSummary` list, same as before.

In [ ]:
summaries = parse_sessions(TARGET)
print(f"\n{len(summaries)} SessionSummary object(s) returned")

## 3. Get the DataFrames directly, for filtering

`events_df` - one row per raw event (same shape as
`pd.read_json(path, lines=True)`).
`sessions_df` - one row per session, grouped from `events_df`. This is
the frame to filter against for picking LLM targets.

In [ ]:
input_context = new_run_context(log_files, MODEL)
events_df = load_events_df(TARGET)
sessions_df = build_session_summaries_df(events_df)
sessions_df.head()

## 4. Technique candidate frequency

Needs `sessions_df` (not `events_df`) - `enrich_session()` requires
per-session command/login/download context together, not individual
event rows.

In [ ]:
print("--- credentials ---")
print(credential_stats(events_df, top_n=10))

print("\n--- source IPs ---")
print(ip_stats(events_df, top_n=10))

print("\n--- commands ---")
print(command_stats(events_df, top_n=10))

print("\n--- downloads ---")
print(download_stats(events_df, top_n=10))

print("\n--- ATT&CK technique candidates ---")
print(technique_stats(sessions_df, top_n=10))

## 5. Filter for LLM targets

Ordinary pandas filtering on `sessions_df`. Example below: sessions
that downloaded at least one file. Adjust the filter to whatever
you've decided is worth the LLM's time based on the stats above -
e.g. repeat source IPs, higher command counts, specific credentials.

In [ ]:
targets_df = sessions_df[sessions_df["files_downloaded"].str.len() > 0]
print(f"{len(targets_df)} session(s) selected for LLM triage")

targets = dataframe_to_summaries(targets_df)

## 6. LLM triage - only the filtered targets

`iter_triage()` yields one result at a time, so each session gets a
progress line AND an audit trail write the moment it's done - not held
back until the whole batch finishes. Change `targets` to `summaries`
here if you want to run the full unfiltered set instead.

### Inspect the selected sessions and captured payloads

Run this before calling Ollama when you want to examine the evidence yourself. The first cell displays every raw event from the selected sessions in timestamp order.

In [ ]:
target_session_ids = set(targets_df["session_id"])
target_events_df = events_df[events_df["session"].isin(target_session_ids)].copy()

timeline_columns = [
	column for column in [
		"timestamp", "session", "src_ip", "eventid", "username",
		"password", "input", "url", "destfile", "outfile",
		"shasum", "message"
	]
	if column in target_events_df.columns
]

target_events_df[timeline_columns].sort_values(["session", "timestamp"])

Cowrie's `outfile` points to its download store on the Pi. Set `COWRIE_DOWNLOAD_DIR` in `.env` to the local folder where you copied that store, for example `D:/wraith-evidence/downloads`. The next cell never executes a captured file: it checks the SHA-256 and previews text only. Binary content is reported without printing it.

In [ ]:
PAYLOAD_PREVIEW_BYTES = 2000
COWRIE_DOWNLOAD_DIR = Path(
	os.getenv("COWRIE_DOWNLOAD_DIR", REPO_ROOT / "data" / "downloads")
)

download_events_df = target_events_df[
	target_events_df["eventid"] == "cowrie.session.file_download"
]

for _, download in download_events_df.iterrows():
	stored_name = Path(str(download.get("outfile", ""))).name
	payload_path = COWRIE_DOWNLOAD_DIR / stored_name

	print(f"\n--- session={download['session']} destfile={download.get('destfile')} ---")
	print(f"Cowrie SHA-256: {download.get('shasum')}")
	print(f"Local path: {payload_path}")

	if not payload_path.is_file():
		print("Payload not copied to COWRIE_DOWNLOAD_DIR")
		continue

	payload = payload_path.read_bytes()
	actual_shasum = hashlib.sha256(payload).hexdigest()
	print(f"Local SHA-256:  {actual_shasum}")
	print(f"Hash matches:   {actual_shasum == download.get('shasum')}")

	preview = payload[:PAYLOAD_PREVIEW_BYTES]
	if b"\x00" in preview:
		print(f"Binary payload ({len(payload)} bytes); text preview skipped")
	else:
		print(preview.decode("utf-8", errors="replace"))

In [ ]:
# Re-run the loading cells after changing TARGET or refreshing local logs.
current_inputs = new_run_context(log_files, MODEL)["input_files"]
if current_inputs != input_context["input_files"]:
	raise RuntimeError("Input logs changed; rerun the loading cells before triage")
run_context = new_run_context(log_files, MODEL, source_files=[
	REPO_ROOT / "src/wraith/parsers/cowrie_parser.py",
	REPO_ROOT / "src/wraith/llm/session_assessment.py",
	REPO_ROOT / "src/wraith/enrichment/attack_mapping.py",
	REPO_ROOT / "src/wraith/enrichment/attack_lookup.py",
	REPO_ROOT / "src/wraith/enrichment/triage.py",
])
run_context["target_session_ids"] = [s.session_id for s in targets]
print(f"Assessment run: {run_context['run_id']}")
all_summaries = dataframe_to_summaries(sessions_df)
results = []
run_start = time.perf_counter()
total = len(targets)

for i, result in enumerate(iter_triage(targets, analyze_session, BASE_URL, MODEL, correlation_summaries=all_summaries), start=1):
	log_batch([result], model=MODEL, run_context=run_context)

	elapsed_total = time.perf_counter() - run_start
	now = datetime.now().strftime("%H:%M:%S")

	print(f"[{now}] ({i}/{total}) {result['session_id']}  method={result['method']}  classification={result['classification']}  elapsed_total={elapsed_total:.1f}s")

	results.append(result)

print(f"\ndone - {total} session(s) in {time.perf_counter() - run_start:.1f}s")

## 7. Review results

In [ ]:
classification_counts = Counter(r["classification"] for r in results)

for classification, count in classification_counts.items():
	print(f"{classification:>12}: {count}")

In [ ]:
flagged_results = [r for r in results if r["classification"] in ("suspicious", "malicious")]
print(f"{len(flagged_results)} session(s) classified suspicious/malicious")

for r in flagged_results:
	print(f"\n--- {r['session_id']}  ({r['src_ip']}) ---")
	print(f"classification={r['classification']}  confidence={r['confidence']}")
	if r["assessment"]:
		print(r["assessment"].model_dump_json(indent=2))
	if r["ioc_history"]:
		ioc = r["ioc_history"]
		print(f"IoC correlation: {len(ioc.related_session_ids)} related session(s) ({ioc.related_session_ids}), any_login_success={ioc.any_login_success}, total_commands={ioc.total_commands}, total_downloads={ioc.total_downloads}")

## 8. Retrieve reviewed payloads

Set `REVIEWED_SESSION_IDS` explicitly after reviewing the timelines and results above. Only captured downloads from those sessions enter the manifest. Empty selection exports no files. The manifest requires Cowrie SHA-256 storage filenames and recorded hashes; incomplete evidence raises an error rather than guessing.

After exporting, run from the project root:

```powershell
python scripts/pi_logs.py payloads --manifest data/payload_manifest.json
```

The default Pi store is `/home/cowrie/cowrie/var/lib/cowrie/downloads`; override with `--payload-dir` if needed. Copies go to `data/downloads/`; verified existing copies are skipped. Failed or mismatched downloads are reported and do not replace existing evidence. Re-run the payload preview cell above after collection (selected sessions must also be in `targets_df`). Nothing executes the payloads or visits their URLs.

Each triage execution has a new run ID and records input/source hashes and the configured model name. Correlation uses all loaded sessions; repeated assessments are intentionally appended. The model name is not an immutable model digest.


In [ ]:
# Use the collection helper from the existing Pi script.
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from pi_logs import export_payload_manifest

REVIEWED_SESSION_IDS = []  # Add reviewed session IDs, e.g. ["session-id"]
manifest_path = REPO_ROOT / "data" / "payload_manifest.json"
manifest = export_payload_manifest(events_df, REVIEWED_SESSION_IDS, manifest_path)
print(f"Exported {len(manifest['files'])} download reference(s) to {manifest_path}")
